# The Rest of the Iceberg — the native-kernel cut

Same exploration as the
[companion notebook](the-rest-of-the-iceberg.ipynb), on the
**Quarb Jupyter kernel**: every cell *is* a query — no `%%` fences,
no quoting — with tab-completion over the mounted tree. Directives
(`%mount`, `%use`, `%docs`, `%translate`, `%python`) manage state.

Setup:

```
pip install "quarb[jupyter]"
python -m quarb.kernel install
```

then open this notebook with the **Quarb** kernel, with
`titanic.csv` and `titanic-birth-records.json` alongside.

In [1]:
%mount titanic.csv titanic-birth-records.json

mounted: titanic+titanic-birth-records (children: /titanic, /titanic-birth-records)


In [2]:
/titanic-birth-records/archives/*:::name

gro
irish_parishes
kyrkobocker
actes_de_naissance
us_state_records
ottoman_nufus
matricula_baptizatorum
metricheskie_knigi

In [3]:
/titanic-birth-records/archives/* | .a(:::name) | //passenger_id @| group($.a) | count | .n | %.

a,n
gro,422
irish_parishes,124
kyrkobocker,166
actes_de_naissance,27
us_state_records,286
ottoman_nufus,80
matricula_baptizatorum,34
metricheskie_knigi,7


## Naming the vocabulary

A `def` names a fragment of query text; invocations expand at parse
time. A cell holding only definitions (and `#` comments) extends
the **session's fragment table** — its names resolve in every later
cell, so the notebook states its vocabulary once, then speaks it.
`&record` is the birth record for the row at hand (`$$` names the
driving row from inside the join condition); `&gregorian` converts
the Russian Empire's Julian dates:

In [4]:
# the exploration's vocabulary
def &record: /titanic-birth-records//*[::passenger_id = $$::PassengerId] ;
def &records: /titanic-birth-records//*[::passenger_id] ;
# Julian to Gregorian, 1900-1917
def &gregorian: | tp("%Y-%m-%d") | ($_ + 12d);

defined: &record, &records, &gregorian


In [5]:
/titanic/row <=>? &record | [not $*1::passenger_id] | ::Name @| [..6]

Vestrom, Miss. Hulda Amanda Adolfina
Hewlett, Mrs. (Mary D Kingcome) 
Vander Planke, Mrs. Julius (Emelia Maria Vandemoortele)
Todoroff, Mr. Lalio
Uruchurtu, Don. Manuel E
Vander Planke, Miss. Augusta Maria

In [6]:
&records | [::passenger_id > 891] @| count

373

## Recovering the missing ages

The eight-branch ladder normalizes eight date conventions into one
birth year — `&gregorian` comes from the vocabulary cell above:

In [7]:
/titanic/row[not ::Age] <=> &record
 | .born(( $*1::year_rumi      ? $*1::year_rumi + 584
         : $*1::quarter        ? ($*1::quarter | s/ Q.*$//) * 1
         : $*1::date           ? ($*1::date | s/^.*\s//) * 1
         : $*1::född           ? ($*1::född | s/-.*$//) * 1
         : $*1::né_le          ? ($*1::né_le | s/-.*$//) * 1
         : $*1::date_of_birth  ? ($*1::date_of_birth | s/-.*$//) * 1
         : $*1::natus          ? ($*1::natus | s/-.*$//) * 1
         : $*1::nata           ? ($*1::nata | s/-.*$//) * 1
         : (($*1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | rec(::Name, "born", $.born, "age_in_1912", 1912 - $.born) @| [..10]

Name,born,age_in_1912
"Moran, Mr. James",1889,23
"Williams, Mr. Charles Eugene",1875,37
"Masselmani, Mrs. Fatima",1877,35
"Emir, Mr. Farred Chehab",1881,31
"O'Dwyer, Miss. Ellen ""Nellie""",1894,18
"Spencer, Mrs. William Augustus (Marie Eugenie)",1888,24
"Glynn, Miss. Mary Agatha",1895,17
"Mamee, Mr. Hanna",1885,27
"Rogers, Mr. William John",1871,41
"Lennon, Mr. Denis",1872,40


## The sisters nobody's ticket reveals

In [8]:
/titanic-birth-records//födda/*
 <=> /titanic/row[::PassengerId = $$::passenger_id]
 <=> /titanic-birth-records//födda/*[::fader = $$::fader
       && ::moder = $$::moder && ::passenger_id > $$::passenger_id]
 | [$*1::Name *= "Mrs."]
 | rec("aboard_as", $*1::Name, "sibling", $*2::namn)

aboard_as,sibling
"Asplund, Mrs. Carl Oscar (Selma Augusta Emilia Johansson)",Gustaf Joel Johansson
"Ahlin, Mrs. Johan (Johanna Persdotter Larsson)",Bengt Edvin Larsson
"Backstrom, Mrs. Karl Alfred (Maria Mathilda Gustafsson)",Anders Vilhelm Gustafsson


## Coming from jq?

`%translate` renders a jq (or XPath, or SQL) query as Quarb text —
useful while the muscle memory transfers:

In [9]:
%translate jq .archives.gro.entries[] | select(.quarter == "1890 Q1") | .surname

/archives/gro/entries/*[/quarter:: = "1890 Q1"]/surname::


## A macro that asks the data

Fragments are textual; *macros* run at expansion time, and the
data-aware form `&name!` may read the mounted dataset while
expanding. The classic showpiece — a pivot table whose columns are
the distinct values of a column, something no fixed query can
spell — mean fare by class, with a margin:

In [10]:
macro &pivot!($col, $of):
      /data/titanic/row | $col @| unique @| sort
        | "| .${$_}(^/titanic/row[${$col} = ${$_}]${$of} @| mean)";
    ^ | &pivot!(::Pclass, ::Fare)
      | .All(^/titanic/row::Fare @| mean) | %.

1,2,3,All
84.15468749999992,20.66218315217391,13.675550101832997,32.2042079685746


## The escape hatch

`%python` cells are ordinary Python — `_` holds the last query
result, `session` is the live session (the fragment table
included, so `session.run` still speaks `&gregorian`), and `.df`
turns a record stream into a DataFrame. The enriched table, then
the finding:

In [11]:
/titanic/row <=>? &record
 | .born(( $*1::year_rumi      ? $*1::year_rumi + 584
         : $*1::quarter        ? ($*1::quarter | s/ Q.*$//) * 1
         : $*1::date           ? ($*1::date | s/^.*\s//) * 1
         : $*1::född           ? ($*1::född | s/-.*$//) * 1
         : $*1::né_le          ? ($*1::né_le | s/-.*$//) * 1
         : $*1::date_of_birth  ? ($*1::date_of_birth | s/-.*$//) * 1
         : $*1::natus          ? ($*1::natus | s/-.*$//) * 1
         : $*1::nata           ? ($*1::nata | s/-.*$//) * 1
         : (($*1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | rec("id", ::PassengerId * 1, ::Sex, "pclass", ::Pclass * 1,
       "survived", ::Survived * 1,
       "age", (( ::Age ? ::Age * 1 : 1912 - $.born )),
       "age_source", (( ::Age ? "manifest" : $.born ? "records" : "unknown" )))
 @| [..5]

id,Sex,pclass,survived,age,age_source
1,male,3,0,22,manifest
2,female,1,1,38,manifest
3,female,3,1,26,manifest
4,female,1,1,35,manifest
5,male,3,0,35,manifest


In [12]:
%python
full = session.run("""
/titanic/row <=>? /titanic-birth-records//*[::passenger_id = $$::PassengerId]
 | .born(( $*1::year_rumi      ? $*1::year_rumi + 584
         : $*1::quarter        ? ($*1::quarter | s/ Q.*$//) * 1
         : $*1::date           ? ($*1::date | s/^.*\s//) * 1
         : $*1::född           ? ($*1::född | s/-.*$//) * 1
         : $*1::né_le          ? ($*1::né_le | s/-.*$//) * 1
         : $*1::date_of_birth  ? ($*1::date_of_birth | s/-.*$//) * 1
         : $*1::natus          ? ($*1::natus | s/-.*$//) * 1
         : $*1::nata           ? ($*1::nata | s/-.*$//) * 1
         : (($*1::data_rozhdeniya | &gregorian) | s/-.*$//) * 1 ))
 | rec("id", ::PassengerId * 1, ::Sex, "pclass", ::Pclass * 1,
       "survived", ::Survived * 1,
       "age", (( ::Age ? ::Age * 1 : 1912 - $.born )),
       "age_source", (( ::Age ? "manifest" : $.born ? "records" : "unknown" )))
""")
df = full.df
print(df.shape)
print(df.groupby("age_source")["survived"].mean().round(3))

(891, 6)
age_source
manifest    0.406
records     0.318
unknown     0.100
Name: survived, dtype: float64


From 177 unknown ages to 20 — and the recovered rows survived at
32% against 41% for the rows the manifest filled in: the
missingness was never random, and the median would have buried
that. The vocabulary accretes, the queries stay speakable, and the
table you finally export has earned its shape.